In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip -q install -U transformers accelerate datasets sentencepiece

import os
import re
import json
import time
import glob
import subprocess
from pathlib import Path

import torch
import pandas as pd
from transformers import AutoProcessor, AutoModelForImageTextToText

In [ ]:
MODEL_CANDIDATES = [
    "/kaggle/input/gemma-4-31b-it-qat-w4a16-ct",
    "/kaggle/input/gemma-4-31b-it-qat-w4a16-ct/1",
    "google/gemma-4-31B-it-qat-w4a16-ct"
]

MODEL_PATH = None

for p in MODEL_CANDIDATES:
    if p.startswith("/kaggle/input"):
        if os.path.exists(p):
            MODEL_PATH = p
            break
    else:
        MODEL_PATH = p

print("MODEL:", MODEL_PATH)

In [ ]:
!pip install -q "compressed-tensors>=0.15.0"

In [ ]:
import compressed_tensors
print(compressed_tensors.__version__)

In [ ]:
%pip install -U "compressed-tensors>=0.15.0"

In [ ]:
import torch
import transformers
import compressed_tensors

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Compressed tensors:", compressed_tensors.__version__)

In [ ]:
%pip uninstall -y transformers
%pip install --no-cache-dir git+https://github.com/huggingface/transformers.git

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "compressed-tensors>=0.15.0"
])

import compressed_tensors
print("compressed-tensors:", compressed_tensors.__version__)

In [ ]:
import transformers
print("Transformers:", transformers.__version__)
print("Path:", transformers.__file__)

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--upgrade",
    "--force-reinstall",
    "git+https://github.com/huggingface/transformers.git"
])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--upgrade",
    "compressed-tensors>=0.15.0"
])

print("Installation complete. Restart the Kaggle session now.")

In [1]:
import transformers
import compressed_tensors

print("Transformers:", transformers.__version__)
print("Transformers path:", transformers.__file__)
print("Compressed tensors:", compressed_tensors.__version__)

Transformers: 5.18.0.dev0
Transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Compressed tensors: 0.19.0


In [2]:
from transformers.models.gemma4.configuration_gemma4 import Gemma4Config

print("Gemma4 model type:", Gemma4Config.model_type)

Gemma4 model type: gemma4


In [3]:
import torch
import transformers
import compressed_tensors

print("Transformers:", transformers.__version__)
print("Compressed tensors:", compressed_tensors.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Transformers: 5.18.0.dev0
Compressed tensors: 0.19.0
CUDA: True
GPU: Tesla T4


In [4]:
from transformers import AutoConfig

MODEL_PATH = "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2"

config = AutoConfig.from_pretrained(MODEL_PATH)

print("Config class:", type(config).__name__)
print("Model type:", config.model_type)

Config class: Gemma4Config
Model type: gemma4


In [5]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("Processor loaded:", type(processor).__name__)

Processor loaded: Gemma4Processor


In [6]:
from transformers import AutoModelForMultimodalLM

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    dtype="auto"
)

model.eval()

print("Model loaded:", type(model).__name__)

Compressing model: 100%|██████████| 410/410 [00:00<00:00, 8930.86it/s]


Loading weights:   0%|          | 0/2009 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 16.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.42 GiB is allocated by PyTorch, and 27.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [7]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

14.424074172973633 GB allocated
14.443359375 GB reserved


In [9]:
import torch
import transformers
import compressed_tensors

print("Transformers:", transformers.__version__)
print("Compressed tensors:", compressed_tensors.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

Transformers: 5.18.0.dev0
Compressed tensors: 0.19.0
CUDA: True
GPU: Tesla T4
Allocated: 14.42 GB
Reserved: 14.44 GB


In [10]:
import torch
import gc

print("=== PYTORCH ===")
print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")

print("\n=== NVIDIA-SMI ===")

=== PYTORCH ===
Allocated: 14.424074172973633 GB
Reserved: 14.443359375 GB

=== NVIDIA-SMI ===


In [11]:
!nvidia-smi

Wed Sep 30 16:35:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             30W /   70W |   14895MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import gc
import torch

cuda_tensors = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            cuda_tensors.append(obj)
    except:
        pass

cuda_tensors = sorted(
    cuda_tensors,
    key=lambda x: x.numel() * x.element_size(),
    reverse=True
)

print("CUDA tensors:", len(cuda_tensors))

for t in cuda_tensors[:20]:
    size_gb = t.numel() * t.element_size() / 1024**3
    print(
        "shape=", tuple(t.shape),
        "dtype=", t.dtype,
        "device=", t.device,
        "size=", round(size_gb, 3), "GB"
    )

CUDA tensors: 1821
shape= (262144, 5376) dtype= torch.bfloat16 device= cuda:0 size= 2.625 GB
shape= (262144, 5376) dtype= torch.bfloat16 device= cuda:0 size= 2.625 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (5376, 2688) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (5376, 2688) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (5376, 2688) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (21504, 672) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape= (5376, 2688) dtype= torch.int32 device= cuda:0 size= 0.054 GB
shape

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [13]:
import gc
import sys
import torch

for name in list(globals()):
    if name not in {"gc", "sys", "torch"}:
        try:
            obj = globals()[name]
            if torch.is_tensor(obj) or hasattr(obj, "parameters"):
                del globals()[name]
        except:
            pass

sys.last_traceback = None
sys.last_value = None
sys.last_type = None

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

Allocated: 14.424 GB
Reserved: 14.443 GB


In [14]:
print([x for x in globals().keys() if "model" in x.lower()])

['MODEL_PATH', 'AutoModelForMultimodalLM']


In [15]:
!nvidia-smi

Wed Sep 30 16:37:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0             30W /   70W |   14895MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i),
        "free:", round(free / 1024**3, 2),
        "GB /",
        round(total / 1024**3, 2),
        "GB"
    )

CUDA: True
GPU count: 2
GPU 0: Tesla T4 free: 0.02 GB / 14.56 GB
GPU 1: Tesla T4 free: 7.77 GB / 14.56 GB


In [17]:
!nvidia-smi



Wed Sep 30 16:38:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             30W /   70W |   14895MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB"
    )

GPU count: 2
GPU 0: 0.02 GB free / 14.56 GB
GPU 1: 7.77 GB free / 14.56 GB


In [19]:
import transformers
import compressed_tensors

print("Transformers:", transformers.__version__)
print("Compressed tensors:", compressed_tensors.__version__)

from transformers.models.gemma4.configuration_gemma4 import Gemma4Config
print("Gemma:", Gemma4Config.model_type)

Transformers: 5.18.0.dev0
Compressed tensors: 0.19.0
Gemma: gemma4


In [20]:
!nvidia-smi --query-gpu=index,memory.used,memory.free,memory.total --format=csv

index, memory.used [MiB], memory.free [MiB], memory.total [MiB]
0, 14895 MiB, 17 MiB, 15360 MiB
1, 6951 MiB, 7961 MiB, 15360 MiB


In [ ]:
import os
import signal

os.kill(os.getpid(), signal.SIGKILL)

In [1]:
!nvidia-smi --query-gpu=index,memory.used,memory.free,memory.total --format=csv

index, memory.used [MiB], memory.free [MiB], memory.total [MiB]
0, 0 MiB, 14912 MiB, 15360 MiB
1, 0 MiB, 14912 MiB, 15360 MiB


In [2]:
from transformers import AutoProcessor

MODEL_PATH = "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2"

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("Processor:", type(processor).__name__)

Processor: Gemma4Processor


In [3]:
from transformers import AutoModelForMultimodalLM

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    device_map="balanced_low_0",
    max_memory={
        0: "13GiB",
        1: "13GiB",
        "cpu": "24GiB"
    },
    offload_folder="/kaggle/working/gemma_offload",
    offload_state_dict=True,
    dtype="auto"
)

model.eval()

print("Loaded:", type(model).__name__)
print(model.hf_device_map)

Compressing model: 100%|██████████| 410/410 [00:00<00:00, 9487.28it/s]


Loading weights:   0%|          | 0/2009 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded: Gemma4ForConditionalGeneration
{'model.vision_tower': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 1, 'model.language_model.layers.9': 1, 'model.language_model.layers.10': 1, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'mo

In [6]:
!vllm serve google/gemma-4-31B-it-qat-w4a16-ct

/bin/bash: line 1: vllm: command not found


In [1]:
!7z i | head -5


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,4 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)


